# DRAGNET — micro-batch: self-citation + set-removal over existing cells

Two one-generation-per-case passes over every attached natural cell that has an enumerated
family: **self-citation** (does the model know which passages its answer relied on — the scale
curve against the 120B anchor) and **set-removal** (is the smallest sufficient set necessary,
and does removing it repair the error). ~10–20 min per 7B cell; the whole batch fits one short
commit.

**Before running:** GPU T4 x2 · Internet on · attach the dataset holding the natural-cell zips
(`dragnet_*`/`scope_*`).

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
os.chdir(WORK)

def fetch(name, url):
    if (WORK / name).exists():
        subprocess.run(['git', '-C', name, 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, name], check=True)

fetch('lineup', 'https://github.com/santoshcheethiralame-dot/LINEUP')
fetch('dragnet', 'https://github.com/santoshcheethiralame-dot/DRAGNET')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', './lineup[gpu]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', './dragnet'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes>=0.46.1'], check=True)
import bitsandbytes
print('[ok] installed; bitsandbytes', bitsandbytes.__version__)

In [ ]:
MODEL_ZOO = {
    'qwen':    'Qwen/Qwen2.5-7B-Instruct',
    'phi':     'microsoft/Phi-3.5-mini-instruct',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
}
LIMIT = 0     # wrong cases per cell per pass (0 = all)

# Stage every attached cell (zips or pre-extracted, with or without a runs/ prefix).
import shutil, zipfile
scratch = Path('_incoming')
for path in sorted(set(Path('/kaggle/input').glob('**/dragnet_*.zip')) | set(Path('/kaggle/input').glob('**/scope_*.zip'))):
    with zipfile.ZipFile(path) as archive:
        archive.extractall(scratch / path.stem)
    print('unzipped', path.name)
sources = set(Path('/kaggle/input').glob('*/**/scenarios.jsonl')) | set(scratch.glob('**/scenarios.jsonl'))
for marker in sorted(sources):
    parent = marker.parent
    if parent.name not in MODEL_ZOO:
        continue
    dest = Path('runs', *parent.parts[-3:])
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(parent, dest)

cells = sorted(
    c for c in Path('runs').glob('*/natural/*')
    if (c / 'mscs.jsonl').exists() and c.name in MODEL_ZOO
)
assert cells, 'no natural cells with mscs.jsonl found -- attach the results dataset'
print('cells:', [str(c) for c in cells])

In [ ]:
os.chdir(WORK / 'dragnet')
for cell in cells:
    model = MODEL_ZOO[cell.name]
    target = WORK / 'dragnet' / 'cellwork' / cell
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        shutil.copytree(WORK / cell, target)
    lim = ['--limit', str(LIMIT)] if LIMIT else []
    for script, name in (('scripts/run_selfcite.py', 'selfcite'), ('scripts/run_removal.py', 'removal')):
        print(f'==== {name}: {cell} ({model}) ====', flush=True)
        subprocess.run([sys.executable, script, '--cell', str(target),
                        '--model', model, '--load-in-4bit', *lim], check=True)
    print(f'[ok] {cell}', flush=True)
os.chdir(WORK)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/dragnet_micro', 'zip', WORK / 'dragnet' / 'cellwork')
print('download dragnet_micro.zip from the Output panel')